# Control Panel 2: Train Model

This notebook trains a CNN on the generated DQD stability diagram dataset.

## Usage
1. Ensure data has been generated using `1_Generate_Data.ipynb`
2. Configure training parameters below
3. Run all cells to train the model

In [ ]:
# Import required modules
import sys
from pathlib import Path

# Add src to path
sys.path.append('src')

from src.config import default_params_5, DATA_DIR, MODEL_DIR
from src.dataloader import create_dataloader, get_dataset_info
from src.model import get_model

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

## Check Dataset

Verify that the dataset exists and check its properties.

In [ ]:
# Dataset path
dataset_path = Path(DATA_DIR) / 'dqd_dataset.h5'

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {dataset_path}. "
        "Please run 1_Generate_Data.ipynb first."
    )

# Get dataset info
info = get_dataset_info(str(dataset_path))
print("Dataset Information:")
for key, value in info.items():
    print(f"  {key}: {value}")

## Configure Training Parameters

In [ ]:
# Training configuration
config = {
    'batch_size': 32,
    'learning_rate': 0.001,
    'epochs': 50,
    'model_type': 'autoencoder',  # 'classifier' or 'autoencoder'
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 4,
    'save_interval': 10,  # Save model every N epochs
}

print(f"Training on device: {config['device']}")
print(f"Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## Create DataLoader

In [ ]:
# Create dataloader
train_loader = create_dataloader(
    str(dataset_path),
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    normalize=True
)

print(f"DataLoader created with {len(train_loader)} batches")
print(f"Total samples: {len(train_loader.dataset)}")

## Initialize Model

In [ ]:
# Create model
model = get_model(model_type=config['model_type'])
model = model.to(config['device'])

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model created: {config['model_type']}")
print(f"Number of trainable parameters: {num_params:,}")
print(f"\nModel architecture:")
print(model)

## Setup Training

In [ ]:
# Loss function and optimizer
if config['model_type'] == 'autoencoder':
    criterion = nn.MSELoss()
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                   factor=0.5, patience=5)

# Create model directory
Path(MODEL_DIR).mkdir(exist_ok=True)

# TensorBoard writer
writer = SummaryWriter(log_dir=f'runs/{config["model_type"]}')

print("Training setup complete!")

## Train Model

In [ ]:
# Training loop
train_losses = []
best_loss = float('inf')

print("Starting training...\n")

for epoch in range(config['epochs']):
    model.train()
    epoch_loss = 0.0
    
    # Progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs = inputs.to(config['device'])
        targets = targets.to(config['device'])
        
        # Forward pass
        optimizer.zero_grad()
        
        if config['model_type'] == 'autoencoder':
            outputs, _ = model(inputs)
        else:
            outputs = model(inputs)
        
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
    
    # Calculate average loss
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Log to tensorboard
    writer.add_scalar('Loss/train', avg_loss, epoch)
    writer.add_scalar('Learning_rate', optimizer.param_groups[0]['lr'], epoch)
    
    # Learning rate scheduling
    scheduler.step(avg_loss)
    
    print(f"Epoch {epoch+1}/{config['epochs']} - Loss: {avg_loss:.6f}")
    
    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, Path(MODEL_DIR) / 'best_model.pth')
        print(f"  → Saved best model (loss: {best_loss:.6f})")
    
    # Save checkpoint periodically
    if (epoch + 1) % config['save_interval'] == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, Path(MODEL_DIR) / f'checkpoint_epoch_{epoch+1}.pth')
        print(f"  → Saved checkpoint")

writer.close()
print("\nTraining complete!")

## Plot Training Progress

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.show()

print(f"Best loss: {best_loss:.6f}")
print(f"Final loss: {train_losses[-1]:.6f}")

## Next Steps

✅ Model training complete!

Now proceed to **3_Test_Model.ipynb** to evaluate and visualize model performance.